In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ============================================================
# Add a subject_id column to exclude_list.csv
# ============================================================
# The manuscript refers to excluded subjects by numeric ID
# (Subject 017, 118, 126, ...), while the exclusion list was written
# with the dataset's original folder names. This cell adds a matching
# subject_id column so a reader can cross-reference the CSV against
# the paper directly. The ID is the class index assigned by the same
# sorted-folder ordering used everywhere in the pipeline.

import os
import pandas as pd

FACE_TEST = '/content/drive/MyDrive/split faces/test'
CSV_IN    = '/content/drive/MyDrive/exclude_list.csv'   # adjust if needed
CSV_OUT   = '/content/drive/MyDrive/exclude_list.csv'   # overwrite in place

# Same class-index map used throughout: sorted subject folders -> index
Code = {c.lower().strip(): i
        for i, c in enumerate(sorted(os.listdir(FACE_TEST)))
        if os.path.isdir(os.path.join(FACE_TEST, c))}

df = pd.read_csv(CSV_IN)

# Identify the column holding the subject/folder name
name_col = None
for cand in ['subject', 'name', 'folder', 'person', 'class']:
    if cand in df.columns:
        name_col = cand
        break
if name_col is None:
    raise ValueError(f"Could not find a subject-name column in {list(df.columns)}")

def to_id(v):
    key = str(v).lower().strip()
    return f"{Code[key]:03d}" if key in Code else "NA"

df.insert(df.columns.get_loc(name_col) + 1, 'subject_id', df[name_col].map(to_id))

df.to_csv(CSV_OUT, index=False)
print(f"Wrote {CSV_OUT} with {len(df)} rows and new 'subject_id' column.")
print("\nExcluded subjects (unique id -> name):")
uniq = df[['subject_id', name_col]].drop_duplicates().sort_values('subject_id')
for _, r in uniq.iterrows():
    print(f"  Subject {r['subject_id']}  =  {r[name_col]}")

# Quick sanity: show the mapping the paper relies on
print("\nCross-check against paper references:")
expected = {'017','041','091','093','118','126'}
found = set(uniq['subject_id']) & expected
print(f"  Paper-referenced IDs present in list: {sorted(found)}")

Wrote /content/drive/MyDrive/exclude_list.csv with 45 rows and new 'subject_id' column.

Excluded subjects (unique id -> name):
  Subject 017  =  ali fadhel
  Subject 029  =  ali mohamad s
  Subject 038  =  ameer basem
  Subject 041  =  atheer saad
  Subject 044  =  aya jeid
  Subject 046  =  banen ahmed
  Subject 047  =  banen hasan
  Subject 055  =  fatma jodat
  Subject 060  =  fras
  Subject 064  =  gafer mohamed
  Subject 083  =  ibrahem hamed
  Subject 091  =  kaher musa
  Subject 093  =  karar ali
  Subject 099  =  maream mahmood
  Subject 103  =  mohamad abdalhaleem
  Subject 115  =  mohamad salah
  Subject 118  =  muheman majeed
  Subject 125  =  mustafa ali
  Subject 126  =  mustafa fathel
  Subject 151  =  shaemaa abdalha
  Subject 160  =  waleed rasheed
  Subject 164  =  yaser amar

Cross-check against paper references:
  Paper-referenced IDs present in list: ['017', '041', '091', '093', '118', '126']
